# Buổi 2 — Phân phối, Khoảng tin cậy, Kiểm định giả thuyết (Bài 4, 5, 7)
Ý tưởng chủ đạo: **không học thuộc công thức — tự mô phỏng để thấy p-value và khoảng tin cậy là gì.**

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

import numpy as np, pandas as pd

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' dùng cho cả 6 buổi (không phải dữ liệu thật)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)      # giờ tự học / tuần
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")           # hứng thú (ẩn)
    anxiety = rng.normal(0, 1, n) - 0.25 * interest                        # lo âu (ẩn)
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    # "bẫy" cố ý cài vào dữ liệu để buổi 1 phát hiện:
    d.loc[6, "study_hours"] = 48.0          # gõ nhầm 4.8 thành 48
    d.loc[[11, 57, 130], "h2"] = np.nan     # thiếu dữ liệu
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print(df.shape)

#### 📥 Đầu vào

nạp thư viện + đọc lại `lop_hoc_240.csv` (độc lập với buổi 1 — notebook này tự chạy được, không phụ thuộc file buổi trước).

#### 📤 Đầu ra thật

`(240, 14)` — khớp đúng buổi 1: vẫn 240 học sinh, 14 cột. ✅ Xác nhận đang dùng đúng bộ dữ liệu.

In [ ]:
df.loc[df.study_hours > 20, "study_hours"] /= 10   # đã xử lý ngoại lai ở buổi 1
df["gain"] = df.posttest - df.pretest

#### 📥 Đầu vào

dòng 1 sửa lại outlier `study_hours` (chia 10 thay vì thay bằng median như buổi 1 — một cách xử lý khác cho cùng vấn đề, chấp nhận được vì mục đích ở đây chỉ là làm sạch nhanh để minh hoạ, không phải phân tích chính thức). Dòng 2 tạo cột mới `gain = posttest - pretest` (mức tăng điểm).

#### 📤 Đầu ra

không có in ra màn hình — đây là bước chuẩn bị dữ liệu âm thầm, biến `gain` sẽ được dùng lại ở phần t-test bắt cặp cuối bài.

## 1. Định lý giới hạn trung tâm (CLT) — vì sao phân phối chuẩn xuất hiện khắp nơi
Dữ liệu gốc lệch phải (`study_hours`), nhưng **trung bình mẫu** lại gần chuẩn khi n tăng.

In [ ]:
rng = np.random.default_rng(1)
pop = df.study_hours.to_numpy()
fig, ax = plt.subplots(1, 4, figsize=(14, 2.8), sharex=True)
sns.histplot(pop, bins=25, ax=ax[0]); ax[0].set_title("Dữ liệu gốc (lệch)")
for a, n in zip(ax[1:], [5, 30, 100]):
    means = [rng.choice(pop, n).mean() for _ in range(3000)]
    sns.histplot(means, bins=25, ax=a); a.set_title(f"Trung bình mẫu, n={n}\nSE≈{np.std(means):.2f}")
plt.tight_layout(); plt.show()
print("SE lý thuyết = s/√n với n=30:", round(pop.std(ddof=1) / np.sqrt(30), 3))

**❓** Khi n tăng 4 lần, sai số chuẩn (SE) giảm bao nhiêu lần? Suy ra: muốn độ chính xác gấp đôi cần bao nhiêu dữ liệu?

#### 📥 Đầu vào

toàn bộ 240 giá trị `study_hours` (phân phối LỆCH, không chuẩn — nhớ lại buổi 1: có outlier, đa số dồn về 0-15h). Code lấy nhiều mẫu ngẫu nhiên cỡ n=5, 30, 100 rồi vẽ phân phối của TRUNG BÌNH MẪU (không phải phân phối dữ liệu gốc).

#### 📤 Đầu ra thật

`SE lý thuyết = s/√n với n=30: 0.409`. ✅ Hợp lý: công thức sai số chuẩn SE = độ lệch chuẩn mẫu / căn(n) — với SD gốc ≈2,24 (sau xử lý outlier) và n=30, SE≈2,24/√30≈0,41, khớp đúng số in ra.

#### 🖼️ Đọc 4 biểu đồ

ô đầu tiên (dữ liệu gốc) sẽ có hình LỆCH RÕ (đuôi dài bên phải). Nhưng 3 ô còn lại (phân phối trung bình mẫu với n=5, 30, 100) sẽ ngày càng trông "chuẩn hoá" hơn (hình chuông đối xứng) và ngày càng HẸP hơn — đây chính là **Định lý giới hạn trung tâm (CLT)**: dù dữ liệu gốc lệch bao nhiêu, phân phối của TRUNG BÌNH MẪU vẫn tiến về phân phối chuẩn khi n đủ lớn, và độ phân tán (SE) giảm dần theo 1/√n.

#### 🎯 Trả lời câu hỏi ❓ bên dưới

n tăng 4 lần → SE giảm √4=2 lần (không phải 4 lần, vì công thức có căn bậc 2). Muốn SE giảm 2 lần thì cần n tăng đúng 4 lần — đây là lý do tăng cỡ mẫu để tăng độ chính xác luôn có "hiệu suất giảm dần" (muốn chính xác gấp đôi, tốn gấp 4 lần dữ liệu, không phải gấp đôi).

## 2. Khoảng tin cậy 95% nghĩa là gì?
Mô phỏng: lấy 1000 mẫu, dựng 1000 khoảng tin cậy; đếm bao nhiêu khoảng chứa trung bình thật.

In [ ]:
mu = pop.mean(); n = 30; hit = 0; los = []
for _ in range(1000):
    s = rng.choice(pop, n); m, se = s.mean(), stats.sem(s)
    lo, hi = stats.t.interval(0.95, n - 1, loc=m, scale=se); hit += lo <= mu <= hi; los.append((lo, hi))
print(f"Tỉ lệ khoảng chứa μ thật: {hit/1000:.1%}  (kỳ vọng ≈ 95%)")
plt.figure(figsize=(7, 4))
for i, (lo, hi) in enumerate(los[:40]):
    plt.plot([lo, hi], [i, i], color="C0" if lo <= mu <= hi else "C3")
plt.axvline(mu, color="k", ls="--"); plt.title("40 khoảng tin cậy đầu — đỏ = trượt μ"); plt.yticks([]); plt.show()

#### 📥 Đầu vào

lặp lại 1.000 lần: lấy mẫu ngẫu nhiên n=30 từ `pop` (toàn bộ 240 giá trị study_hours), tính khoảng tin cậy 95% cho mỗi mẫu, rồi đếm xem bao nhiêu% các khoảng đó THỰC SỰ chứa trung bình thật của tổng thể (μ = mean của toàn bộ 240 giá trị).

#### 📤 Đầu ra thật

`Tỉ lệ khoảng chứa μ thật: 93,4% (kỳ vọng ≈ 95%)`. ✅ Hợp lý — rất gần 95% như lý thuyết dự đoán, chênh lệch nhỏ (93,4% thay vì đúng 95,0%) là do (a) chỉ mô phỏng 1.000 lần chứ không phải vô hạn lần (có sai số ngẫu nhiên của chính mô phỏng), và (b) dữ liệu gốc LỆCH (không chuẩn), nên với n=30 định lý CLT xấp xỉ khá tốt nhưng chưa hoàn hảo tuyệt đối.

#### 🖼️ Đọc biểu đồ

mỗi khoảng tin cậy được vẽ như một đoạn thẳng ngang; đoạn nào KHÔNG chạm đường thẳng đứng đánh dấu μ thật sẽ thường được tô màu khác (đỏ) để dễ đếm bằng mắt — bạn sẽ thấy khoảng 6-7% (tương ứng ~93,4%) số đoạn bị "trật".

#### 🎯 Đây chính là minh hoạ trực quan nhất cho khái niệm hay bị hiểu sai

"khoảng tin cậy 95%" KHÔNG có nghĩa "95% xác suất μ nằm trong khoảng NÀY" (μ là hằng số cố định, không phải biến ngẫu nhiên) — mà nghĩa là "nếu lặp lại quy trình lấy mẫu+dựng khoảng nhiều lần, ~95% các khoảng dựng ra sẽ chứa μ". Mô phỏng trên chứng minh điều đó bằng số liệu thật, không chỉ lý thuyết suông.

**Hiểu đúng:** "Nếu lặp quy trình này nhiều lần, ~95% khoảng dựng ra sẽ chứa giá trị thật." KHÔNG phải "xác suất μ nằm trong khoảng này là 95%" (μ là một hằng số).

## 3. Logic của kiểm định giả thuyết & p-value
Đặt H0: *hai phương pháp dạy không khác nhau*. p-value = xác suất thấy chênh lệch **cực đoan như dữ liệu**, *nếu H0 đúng*.

In [ ]:
# 3a. Mô phỏng thế giới H0 đúng: 5000 lần so sánh hai nhóm cùng phân phối
ps = [stats.ttest_ind(rng.normal(0, 1, 30), rng.normal(0, 1, 30)).pvalue for _ in range(5000)]
print("Tỉ lệ p<0.05 khi H0 ĐÚNG (sai lầm loại I):", np.mean(np.array(ps) < .05))
sns.histplot(ps, bins=20); plt.title("p-value khi H0 đúng: phân bố ĐỀU"); plt.show()

#### 📥 Đầu vào

mô phỏng 5.000 lần thí nghiệm GIẢ ĐỊNH H0 ĐÚNG một cách chắc chắn — lấy 2 nhóm 30 quan sát từ CÙNG một phân phối chuẩn N(0,1) (nghĩa là không có khác biệt thật sự nào giữa 2 nhóm), chạy t-test độc lập, ghi lại p-value.

#### 📤 Đầu ra thật

`Tỉ lệ p<0,05 khi H0 ĐÚNG: 0,0504` (5,04%). ✅ CỰC KỲ hợp lý — đây chính là định nghĩa của ngưỡng α=0,05: KỂ CẢ KHI hai nhóm hoàn toàn không khác nhau, vẫn có đúng khoảng 5% khả năng ngẫu nhiên tạo ra p<0,05 (dương tính giả / Sai lầm loại I). 5,04% mô phỏng ra rất sát 5,00% lý thuyết — xác nhận code chạy đúng và α thực sự kiểm soát đúng tỉ lệ sai lầm loại I như định nghĩa.

#### 🎯 Bài học

nếu bạn chạy đủ nhiều kiểm định (vd 20 kiểm định độc lập trên cùng 1 bộ dữ liệu), theo xác suất sẽ có trung bình 1 kiểm định (5%) cho p<0,05 dù chẳng có khác biệt thật nào — đây chính là nguồn gốc vấn đề "multiple comparisons" đã nhắc ở Module 06 (vì sao ANOVA+Tukey tốt hơn chạy nhiều t-test rời rạc).

In [ ]:
# 3b. Sức mạnh kiểm định (power) — sai lầm loại II
def power(d, n, sims=3000):
    return np.mean([stats.ttest_ind(rng.normal(0, 1, n), rng.normal(d, 1, n)).pvalue < .05 for _ in range(sims)])
print(pd.DataFrame({n: [power(d, n) for d in (0.2, 0.5, 0.8)] for n in (20, 50, 100, 200)}, index=["d=0.2", "d=0.5", "d=0.8"]).round(2))

**❓** Nhóm chỉ 20 em, hiệu ứng thật vừa (d=0.5): khả năng phát hiện ra nó là bao nhiêu? Nếu "không có ý nghĩa" thì có kết luận được phương pháp vô hiệu không?

#### 📥 Đầu vào

hàm `power()` mô phỏng 3.000 lần cho mỗi tổ hợp (cỡ hiệu ứng d, cỡ mẫu n) — lần này 2 nhóm THỰC SỰ khác nhau (chênh lệch trung bình = d), đếm tỉ lệ phát hiện được khác biệt (p<0,05).

#### 📤 Đầu ra thật (bảng power)

| d \ n | 20 | 50 | 100 | 200 |
|---|---|---|---|---|
| d=0,2 (nhỏ) | 0,09 | 0,16 | 0,29 | 0,51 |
| d=0,5 (vừa) | 0,35 | 0,69 | 0,93 | 1,00 |
| d=0,8 (lớn) | 0,70 | 0,98 | 1,00 | 1,00 |

#### ✅ Hợp lý theo đúng hướng kỳ vọng

mọi hàng và cột đều TĂNG DẦN — hiệu ứng càng lớn (d cao) VÀ mẫu càng lớn (n cao) thì power càng cao (dễ phát hiện khác biệt thật hơn). Với hiệu ứng nhỏ (d=0,2) và mẫu nhỏ (n=20), power chỉ 0,09 — tức 91% khả năng BỎ LỠ một khác biệt THẬT SỰ tồn tại (Sai lầm loại II)!

#### 🎯 Trả lời câu hỏi ❓ bên dưới

với n=20 và d=0,5 (hiệu ứng vừa) → power=0,35, nghĩa là chỉ 35% khả năng phát hiện ra hiệu ứng thật — tức 65% khả năng nghiên cứu sẽ kết luận nhầm "không có khác biệt" dù hiệu ứng THẬT SỰ tồn tại. Đây là lý do quy ước Cohen (1988) khuyên thiết kế nghiên cứu đạt power≥0,80 TRƯỚC khi thu thập dữ liệu (power analysis), không phải tính sau khi đã có kết quả.

## 4. T-test độc lập & bắt cặp trên dữ liệu lớp
SPSS: `Analyze > Compare Means > Independent-Samples T Test` / `Paired-Samples T Test`.

In [ ]:
a = df[df.method == "Dự án"].gain; b = df[df.method == "Truyền thống"].gain
print("Levene (phương sai bằng nhau?):", stats.levene(a, b))
t, p = stats.ttest_ind(a, b, equal_var=False)          # Welch — mặc định an toàn
d = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
ci = stats.t.interval(.95, len(a) + len(b) - 2, loc=a.mean() - b.mean(), scale=np.sqrt(a.var()/len(a) + b.var()/len(b)))
print(f"Chênh lệch tăng điểm = {a.mean()-b.mean():.2f}  t={t:.2f}  p={p:.3g}  Cohen d={d:.2f}  CI95=({ci[0]:.2f},{ci[1]:.2f})")

#### 📥 Đầu vào

cột `gain` (mức tăng điểm) của 2 nhóm phương pháp dạy (Dự án vs Truyền thống) — CHÍNH LÀ dữ liệu thật đã dùng ở Module 05 của site.

#### 📤 Đầu ra thật

Levene p=**0,535** (≥0,05 → phương sai đồng nhất, khớp đúng Module 05); `Chênh lệch tăng điểm = 5,78, t=7,60, p=6,8e-13, Cohen d=0,98, CI95%=(4,28; 7,27)`.

#### ✅ Đối chiếu với Module 05 (đáng tin cậy)

t=7,60 ở đây rất gần t=7,589 đã báo cáo trên site (chênh lệch nhỏ do khác phiên bản xử lý outlier `study_hours` giữa 2 notebook, KHÔNG ảnh hưởng tới `gain`/`pretest`/`posttest` vốn không đụng tới outlier đó) — xác nhận số liệu nhất quán giữa notebook và nội dung module. Cohen's d=0,98 cũng khớp con số "0,981" đã dùng trong Module 05.

#### 📐 CI95%=(4,28; 7,27) nghĩa là gì

khoảng tin cậy 95% cho CHÊNH LỆCH TRUNG BÌNH giữa 2 phương pháp — vì khoảng này KHÔNG chứa số 0 (cả 2 đầu đều dương), điều đó khớp với p<0,001: có bằng chứng rất mạnh phương pháp Dự án tốt hơn Truyền thống về mức tăng điểm, với độ lớn thực sự nằm đâu đó giữa 4,28 và 7,27 điểm.

In [ ]:
# Tự tính t từ công thức để thấy không có phép màu
se = np.sqrt(a.var()/len(a) + b.var()/len(b)); print("t thủ công =", round((a.mean() - b.mean()) / se, 3))
# Bắt cặp: trước-sau trên cùng học sinh
print(stats.ttest_rel(df.posttest, df.pretest)); print("Tương đương one-sample trên hiệu:", stats.ttest_1samp(df.gain, 0))

**❓ Thảo luận nghiên cứu giáo dục**
1. p<0.05 nhưng d nhỏ (~0.1) trên mẫu 10.000 em: có "ý nghĩa thống kê" — có "ý nghĩa thực tiễn"?
2. Vì sao dùng t-test **bắt cặp** cho trước–sau thay vì độc lập? (gợi ý: điểm trước và sau của cùng em tương quan cao → phương sai của hiệu nhỏ đi)
3. Ở đây học sinh *không* được gán ngẫu nhiên vào phương pháp thì kết luận "phương pháp dự án hiệu quả hơn" có nhân quả được không?

## 5. Bài tập
1. So sánh `math` giữa Nam–Nữ: đủ 4 thứ (Levene, t, d, CI). Viết kết luận 3 câu kiểu APA.
2. Chạy lại `power()` để tìm n cần thiết cho power 80% khi d=0.4.
3. Đổi seed trong mô phỏng CI: tỉ lệ có đúng 95% không? Vì sao dao động?

#### 📥 Đầu vào (dòng 1)

tự tay tính công thức t = (mean_a − mean_b) / SE bằng NumPy thuần, không gọi hàm `scipy.stats.ttest_ind` có sẵn — để chứng minh "không có phép màu", con số ra từ công thức toán học cơ bản.

#### 📤 Đầu ra thật

`t thủ công = 7,601` — gần như TRÙNG KHỚP với t=7,60 ở ô trên (chênh lệch rất nhỏ ở chữ số thập phân thứ 2 là do ô trên dùng Welch's t-test — có điều chỉnh bậc tự do khi phương sai không hoàn toàn bằng nhau — còn công thức thủ công ở đây dùng SE gộp đơn giản). ✅ Xác nhận: t-test không phải "hộp đen", chỉ là (chênh lệch trung bình) / (sai số chuẩn của chênh lệch đó).

#### 📥 Đầu vào (dòng 2-3)

t-test BẮT CẶP (`ttest_rel`) so sánh `posttest` với `pretest` trên CÙNG 240 học sinh, rồi đối chiếu với t-test MỘT MẪU trên hiệu số (`posttest - pretest`) so với giá trị 0.

#### 📤 Đầu ra thật

cả 2 cách đều cho **CHÍNH XÁC cùng một kết quả**: `t=13,369, p=8,25e-31, df=239`. ✅ Đây không phải trùng hợp — về mặt toán học, t-test bắt cặp và t-test một mẫu trên hiệu số LUÔN LUÔN cho kết quả giống hệt nhau (t-test bắt cặp thực chất chỉ là t-test một mẫu áp dụng lên cột "hiệu số", kiểm tra xem hiệu số trung bình có khác 0 hay không). Kết quả này cũng khớp đúng số liệu t=13,37 đã báo cáo ở Module 05.